# Local and open models
**Ollama in depth · where open weights come from · quantization · tool calling on small models · local vs hosted speed**

Before starting: `git pull`, `uv sync` (no new packages). You'll need Ollama running with at least two local models pulled — see the pre-read.

## 1. Setup

In [1]:
import sys
sys.path[:0] = [".", "module01", "../module01"]

import os

from local_models_utils import (
    model_size_gb,
    fits_on,
    laptop_options,
    load_local_catalog,
    probe_tool_calling,
    speed_table,
)

from providers_utils import groq_client, ollama_client, ask
from landscape_utils import time_stream, PROMPT


# -----------------------------
# Groq setup
# -----------------------------
groq = groq_client()

GROQ_MODEL = os.getenv("MODEL")

print("Groq model:", GROQ_MODEL)


# -----------------------------
# Ollama setup
# -----------------------------
try:
    ollama = ollama_client()

    local_models = [m.id for m in ollama.models.list().data]

    print("Ollama models available:", local_models)

except Exception as e:
    ollama = None
    local_models = []

    print("Ollama not reachable:", e)


# -----------------------------
# Select the Ollama model
# -----------------------------
OLLAMA_MODEL = "llama3.2:3b"

if ollama and OLLAMA_MODEL in local_models:
    print("Ollama model selected:", OLLAMA_MODEL)
elif ollama:
    print(f"Warning: {OLLAMA_MODEL} is not installed.")
    print("Available models:", local_models)

Groq model: openai/gpt-oss-20b
Ollama models available: ['llama3.2:3b']
Ollama model selected: llama3.2:3b


## 2. Where open models come from: Hugging Face
Ollama does not train models — it packages and runs models published on hubs like Hugging Face, the same place Groq's hosted open-weight models originate from. The names differ slightly: a model on Hugging Face as `meta-llama/Llama-3.2-3B-Instruct` becomes `llama3.2:3b` as an Ollama tag, and `openai/gpt-oss-20b` on Groq is the same underlying weights as `gpt-oss:20b` if you pulled it locally.

*Question 1 — If a Hugging Face model page says "gated: request access", what does that tell you about its licence?*

In [2]:
# print(load_local_catalog())

catalog = load_local_catalog()

for m in catalog:
    size = model_size_gb(m["params_billion"], m["precision"])
    print(
        f"{m['name']:20} "
        f"Size: {size:5.1f} GB | "
        f"Tool calling: {m['tool_calling']}  | "
        f"{m['notes']}"
    )

llama3.2:1b          Size:   0.5 GB | Tool calling: False  | Smallest useful chat model. Fast on anything, weak reasoning.
llama3.2:3b          Size:   1.5 GB | Tool calling: False  | The course default local model. No reliable tool calling.
qwen2.5:7b           Size:   3.5 GB | Tool calling: True  | Supports tool calling reasonably well for its size.
llama3.1:8b          Size:   4.0 GB | Tool calling: True  | Good general-purpose local model if RAM allows.
gpt-oss:20b (local)  Size:  10.0 GB | Tool calling: True  | Same weights as the hosted Groq default. Needs a capable laptop or a small GPU.
gpt-oss:120b (local) Size:  60.0 GB | Tool calling: True  | Needs a GPU server, not a laptop. Listed for comparison only.


## 3. Quantization: what it actually costs in memory
Every row below is the same family of maths you saw in Session 10: parameters x bytes-per-parameter. The only new idea is precision — how many bytes each parameter uses.

In [3]:
print(f"{'precision':10} {'bytes/param':>12} {'a 20B model':>14}")

bytes_per_param = {
    "fp32": 4,
    "fp16": 2,
    "int8": 1,
    "int4": 0.5,
}

for precision in ("fp32", "fp16", "int8", "int4"):
    size = model_size_gb(20, precision)
    bpp = bytes_per_param[precision]

    print(f"{precision:10} {bpp:>10} {size:>12.1f} GB")

precision   bytes/param    a 20B model
fp32                4         80.0 GB
fp16                2         40.0 GB
int8                1         20.0 GB
int4              0.5         10.0 GB


In [5]:
# Which catalog models fit YOUR laptop? Edit this to your real RAM.

# Set this to your laptop's actual RAM
MY_RAM_GB = 16

fits = laptop_options(MY_RAM_GB)

print(
    f"Models whose estimated weights fit within a {MY_RAM_GB} GB laptop "
    f"(actual inference memory usage will be higher):"
)

for m in fits:
    size = model_size_gb(
        m["params_billion"],
        m["precision"]
    )

    print(
        f"  {m['name']:22} "
        f"Size:{size:5.1f} GB   "
        f"tool calling: {m['tool_calling']}"
    )

Models whose estimated weights fit within a 16 GB laptop (actual inference memory usage will be higher):
  llama3.2:1b            Size:  0.5 GB   tool calling: False
  llama3.2:3b            Size:  1.5 GB   tool calling: False
  qwen2.5:7b             Size:  3.5 GB   tool calling: True
  llama3.1:8b            Size:  4.0 GB   tool calling: True
  gpt-oss:20b (local)    Size: 10.0 GB   tool calling: True


*Question 2 — Why does the table use int4 sizes for the catalog instead of fp16, when fp16 is what the model was trained at?*

## 4. Does a small local model support tool calling?
Not every small model handles function calling reliably — some will just talk about calling the tool instead of actually emitting a structured call. This matters directly for Module 6: an agent that can't reliably call tools can't be an agent.

In [4]:
# Models you want to test for tool calling
models_to_test = [
    OLLAMA_MODEL,
    # "qwen2.5:7b",
    # "llama3.1:8b",
]

if ollama and local_models:

    for name in models_to_test:

        if name not in local_models:
            print(f"{name}: model not installed")
            continue

        r = probe_tool_calling(ollama, name)

        print(
            f"{name:20} "
            f"supports_tool_calling={r['supports_tool_calling']}"
        )

        if not r["supports_tool_calling"]:
            print(
                "   said instead:",
                (r.get("raw_text") or "")[:150]
            )

else:
    print("Skipped (Ollama not running)")

llama3.2:3b          supports_tool_calling=True


*Question 3 — If your local model fails the tool-calling probe, what are your two options for Module 6's agent labs?*

## 5. Local vs hosted: measure, don't guess
Same prompt, same checkable-answer question from Session 10, sent to Groq and to your local model. Read your own numbers — they depend on your laptop, not on a published benchmark.

In [5]:
# Compare Groq and Ollama using the same benchmark prompt
# and explicitly selected models.

OLLAMA_MODEL = "llama3.2:3b"

results = {}

# -----------------------------
# Groq benchmark
# -----------------------------
try:
    results["groq"] = time_stream(
        groq,
        GROQ_MODEL  # type: ignore
    )
except Exception as e:
    print("Groq failed:", e)


# -----------------------------
# Ollama benchmark
# -----------------------------
if ollama and OLLAMA_MODEL in local_models:
    try:
        results["ollama"] = time_stream(
            ollama,
            OLLAMA_MODEL
        )
    except Exception as e:
        print("Ollama failed:", e)
else:
    print(f"Ollama model not available: {OLLAMA_MODEL}")


# -----------------------------
# Display results
# -----------------------------
if results:
    print(f"Groq model:   {GROQ_MODEL}")
    print(f"Ollama model: {OLLAMA_MODEL}")
    print()

    print(speed_table(results))

Groq model:   openai/gpt-oss-20b
Ollama model: llama3.2:3b

provider     TTFT s   total s   tokens    tok/s
groq           1.41      1.55      919    594.6
ollama         3.70     14.16      146     10.3


*Question 4 — Your laptop's tokens/second is probably 5-15x slower than Groq's. For which of the three jobs from Session 10 (FAQ bot, document reader, agent-assist) does that gap matter least, and why?*

## 6. Reflection
Answer questions 1-4 above, one or two lines each.

1. **Gated access** means the model is not freely accessible under the normal terms; you must request and receive access before downloading or using it.

2. **int4 uses much less memory** because each parameter uses only 0.5 bytes, compared with 2 bytes for fp16. This makes the model practical to run locally while reducing memory requirements.

3. If the local model fails tool calling, I can **use a different local model that supports reliable tool calling**, such as Qwen 2.5 7B or Llama 3.1 8B. Alternatively, I can **use a hosted model** that supports tool calling for the Module 6 agent labs.

4. The speed gap matters least for the **document reader**, because document analysis is generally less latency-sensitive than a live agent-assist system. The slower local model can still be useful when response time is not critical.

Then run the checkpoint: `uv run pytest tests/test_s12.py`